# ============================================================
# AI COMPLIANCE AGENT
# A RAG-based data protection and privacy risk analyst agent for university
# Built with LangChain + Groq (LLaMA 3.3) + FAISS + Streamlit
# ============================================================


## Cell 1: Install all required packages


In [ ]:
# Run this once to set up your Python environment
# ───────────────────────────────────────────────────────────
!pip install -qU \
    langchain \
    langchain-core \
    langchain-community \
    langchain-text-splitters \
    langchain-groq \
    faiss-cpu \
    pypdf \
    streamlit \
    pyngrok


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.6/113.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 90.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.8/173.8 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 97.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently 

## Cell 2: Import core LangChain components


In [ ]:
# These handle: LLM calls, prompt formatting, and output parsing
# ───────────────────────────────────────────────────────────
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


## Cell 3: Set up API key


In [ ]:
# Groq provides fast, free LLM inference
# Get your key at: https://console.groq.com/keys
# In Colab: add it via the 🔑 Secrets panel on the left sidebar
# ───────────────────────────────────────────────────────────
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')


## Cell 4: Initialize the language model


In [ ]:
# We use LLaMA 3.3 70B — a powerful open-source model
# temperature=0 means deterministic, consistent answers
# ───────────────────────────────────────────────────────────
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)


## Cell 5: Download source documents from Google Drive


In [ ]:
# These PDFs are the knowledge base for the AI agent:
#   - UU PDP   : Indonesian Personal Data Protection Law
#   - UU ITE   : Electronic Information & Transactions Law
#   - UU ITE Amendments I & II
#   - ALU Regulations : University internal policies
# ───────────────────────────────────────────────────────────
import gdown
import os

# Create a local folder to store the downloaded PDFs
os.makedirs("documents", exist_ok=True)

# Map each filename to its Google Drive file ID
documents = {
    "UU_PDP.pdf":             "1gwzDwZZoqorirb9XXRE-acXHY9URYIrB",
    "UU_ITE.pdf":             "1DSdMaL2cJGO4__m2CbB1MKDRDnsBb56M",
    "UU_ITE_AmandemenI.pdf":  "1P9C3y6TG98CK9ObB_Ji-nIwP9c_2YfYw",
    "UU_ITE_AmandemenII.pdf": "1TLL4pw8Bk1Q3BOTn7EHOqtRd2X4nsRW4",
    "ALU_Regulations.pdf":    "1pcHYYEUYSdwYPS1Y9DrdWEbqqm0yJ1uk",
}

# Human-readable labels for each document
doc_labels = {
    "UU_PDP.pdf":             "Personal Data Protection Law (UU PDP)",
    "UU_ITE.pdf":             "Electronic Information & Transactions Law (UU ITE)",
    "UU_ITE_AmandemenI.pdf":  "UU ITE Amendment I",
    "UU_ITE_AmandemenII.pdf": "UU ITE Amendment II",
    "ALU_Regulations.pdf":    "ALU University Regulations",
}

print("⏳ Downloading legal documents...\n")
downloaded = []

for filename, file_id in documents.items():
    output_path = f"documents/{filename}"

    # Skip download if file already exists locally
    if os.path.exists(output_path):
        print(f"⏭️  {filename} already exists, skipping.")
        downloaded.append(output_path)
        continue

    try:
        url = f"https://drive.google.com/uc?id={file_id}"
        gdown.download(url, output_path, quiet=False)

        if os.path.exists(output_path):
            size_kb = os.path.getsize(output_path) / 1024
            print(f"✅ {filename} downloaded ({size_kb:.1f} KB)")
            downloaded.append(output_path)
        else:
            print(f"❌ Failed to download {filename}")
    except Exception as e:
        print(f"❌ Error downloading {filename}: {e}")

print(f"\n📂 Successfully downloaded: {len(downloaded)} of {len(documents)} documents")


⏳ Downloading legal documents...



Downloading...
From: https://drive.google.com/uc?id=1gwzDwZZoqorirb9XXRE-acXHY9URYIrB
To: /content/documents/UU_PDP.pdf
100%|██████████| 488k/488k [00:00<00:00, 28.9MB/s]


✅ UU_PDP.pdf downloaded (476.2 KB)


Downloading...
From: https://drive.google.com/uc?id=1DSdMaL2cJGO4__m2CbB1MKDRDnsBb56M
To: /content/documents/UU_ITE.pdf
100%|██████████| 274k/274k [00:00<00:00, 16.5MB/s]


✅ UU_ITE.pdf downloaded (267.6 KB)


...

✅ UU_ITE_AmandemenI.pdf downloaded (1586.9 KB)


Downloading...
From: https://drive.google.com/uc?id=1TLL4pw8Bk1Q3BOTn7EHOqtRd2X4nsRW4
To: /content/documents/UU_ITE_AmandemenII.pdf
100%|██████████| 2.18M/2.18M [00:00<00:00, 14.9MB/s]


✅ UU_ITE_AmandemenII.pdf downloaded (2127.3 KB)


Downloading...
From: https://drive.google.com/uc?id=1pcHYYEUYSdwYPS1Y9DrdWEbqqm0yJ1uk
To: /content/documents/ALU_Regulations.pdf
100%|██████████| 587k/587k [00:00<00:00, 15.4MB/s]

✅ ALU_Regulations.pdf downloaded (572.8 KB)

📂 Successfully downloaded: 5 of 5 documents


## Cell 6: Load and chunk all PDFs


In [ ]:
# RAG (Retrieval-Augmented Generation) works by:
#   1. Breaking documents into small overlapping "chunks"
#   2. Converting each chunk to a vector (embedding)
#   3. At query time, finding the most relevant chunks
#   4. Passing those chunks as context to the LLM
# ───────────────────────────────────────────────────────────
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("📖 Reading PDF documents...\n")

all_documents = []  # Will hold every page from every PDF

for filename, label in doc_labels.items():
    filepath = f"documents/{filename}"

    if not os.path.exists(filepath):
        print(f"⚠️  {filename} not found, skipping.")
        continue

    try:
        loader = PyPDFLoader(filepath)
        pages = loader.load()

        # Tag each page with its source document for citation later
        for page in pages:
            page.metadata["source_name"] = label     # e.g. "UU PDP"
            page.metadata["source_file"] = filename  # e.g. "UU_PDP.pdf"

        all_documents.extend(pages)
        print(f"✅ {label}: {len(pages)} pages loaded")

    except Exception as e:
        print(f"❌ Failed to read {filename}: {e}")

print(f"\n📊 Total pages across all documents: {len(all_documents)}")

# Split each page into overlapping chunks so context isn't cut off mid-sentence
print("\n✂️  Splitting documents into chunks...")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,        # Max 1000 characters per chunk
    chunk_overlap=200,      # 200-character overlap to preserve context at boundaries
    length_function=len,
    # Try to split at paragraph/article boundaries first before cutting mid-sentence
    separators=["\n\n", "\n", "Article ", "Section ", ". ", " "]
)

chunks = text_splitter.split_documents(all_documents)

print(f"✅ Total chunks created: {len(chunks)}")
print(f"\n📝 Sample of first chunk:")
print("-" * 50)
print(f"Source : {chunks[0].metadata.get('source_name', 'Unknown')}")
print(f"Page   : {chunks[0].metadata.get('page', 'Unknown')}")
print(f"Content: {chunks[0].page_content[:300]}...")


📖 Reading PDF documents...

✅ Personal Data Protection Law (UU PDP): 38 pages loaded
✅ Electronic Information & Transactions Law (UU ITE): 38 pages loaded
✅ UU ITE Amendment I: 21 pages loaded
✅ UU ITE Amendment II: 39 pages loaded
✅ ALU University Regulations: 6 pages loaded

📊 Total pages across all documents: 142

✂️  Splitting documents into chunks...
✅ Total chunks created: 318

📝 Sample of first chunk:
--------------------------------------------------
Source : Personal Data Protection Law (UU PDP)
Page   : 0
Content: UNDANG-UNDANG REPUBLIK INDONESIA
NOMOR 27 TAHUN 2022
TENTANG
PELINDUNGAN DATA PRIBADI
DENGAN RAHMAT TUHAN YANG MAHA ESA
PRESIDEN REPUBLIK INDONESIA,
Menimbang:
a.
bahwa pelindungan data pribadi merupakan salah satu hak asasi manusia yang merupakan bagian dari 
pelindungan diri pribadi maka perlu dib...


## Cell 7: Create vector embeddings


In [ ]:
# Embeddings convert text into numerical vectors so we can
# search by semantic meaning, not just keyword matching.
# all-MiniLM-L6-v2 is small, fast, and free (no API key needed)
# ───────────────────────────────────────────────────────────
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


/tmp/ipykernel_35718/1790529777.py:7: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## Cell 8: Build and save the FAISS vector store


In [ ]:
# FAISS is a fast similarity search library by Facebook
# We index ALL chunks (not just the last doc) to enable
# cross-document retrieval across all 5 legal sources
# ───────────────────────────────────────────────────────────
from langchain_community.vectorstores import FAISS

print("🔨 Building FAISS vector index from all document chunks...")

# Index every chunk from all documents
vectorstore = FAISS.from_documents(chunks, embeddings)

# Save to disk so we don't need to rebuild on every run
vectorstore.save_local("faiss_index")

print(f"✅ Vector store built with {len(chunks)} chunks and saved to 'faiss_index/'")


🔨 Building FAISS vector index from all document chunks...
✅ Vector store built with 318 chunks and saved to 'faiss_index/'


## Cell 9: Configure the retriever


In [ ]:
# MMR (Maximal Marginal Relevance) retrieval balances:
#   - Relevance  : how closely the chunk matches the query
#   - Diversity  : avoids returning 10 near-identical chunks
# fetch_k=50 → consider 50 candidates, return best 10
# ───────────────────────────────────────────────────────────
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 10,       # Number of chunks returned to the LLM
        "fetch_k": 50  # Candidate pool size for MMR scoring
    }
)


## Cell 10: Build the retrieval function


In [ ]:
# This function queries the vector store and formats the
# retrieved chunks into a structured context block.
# It deduplicates results and clearly labels each source.
# ───────────────────────────────────────────────────────────
def retrieve_docs(question: str) -> str:
    """
    Query the vector store with a user question and return
    a formatted string of all relevant document excerpts.

    Args:
        question: The user's query about a privacy scenario

    Returns:
        A formatted multi-source context string for the LLM
    """
    # Get the top-k most relevant chunks from the index
    retrieved_docs = retriever.invoke(question)

    # Track which unique chunks we've already added (dedup)
    seen = set()
    context_parts = []

    # Group chunks by their source document for clarity
    sources_used = {}

    for doc in retrieved_docs:
        source_file = doc.metadata.get("source_file", "Unknown")
        source_name = doc.metadata.get("source_name", "Unknown")
        page_num    = doc.metadata.get("page", "?")
        content     = doc.page_content.strip()

        # Create a fingerprint to avoid duplicate content
        unique_key = f"{source_file}:{content[:100]}"

        if unique_key not in seen:
            seen.add(unique_key)

            # Group by document name
            if source_name not in sources_used:
                sources_used[source_name] = []

            sources_used[source_name].append(
                f"  [Page {page_num}]\n  {content}"
            )

    # Format the context with clear section headers per document
    for source_name, excerpts in sources_used.items():
        context_parts.append(
            f"{'='*60}\n"
            f"📄 SOURCE: {source_name}\n"
            f"{'='*60}\n"
            + "\n\n".join(excerpts)
        )

    return "\n\n".join(context_parts)


## Cell 11: Define the analysis prompt


In [ ]:
# This prompt instructs the LLM to act as a university
# data privacy compliance officer and produce a structured
# risk analysis report citing all relevant legal sources.
# ───────────────────────────────────────────────────────────
PRIVACY_ANALYST_PROMPT = """
You are an AI Agent acting as a Data Privacy & Compliance Analyst for a university in Indonesia.

Your role is to:
1. Analyze the described situation or question carefully
2. Assess the privacy/data security risk level
3. Identify ALL relevant articles and clauses from the provided legal documents
4. Cross-reference findings across MULTIPLE documents when applicable
5. Provide concrete, actionable recommendations

LEGAL DOCUMENT CONTEXT (retrieved from vector database):
{context}

SITUATION TO ANALYZE:
{question}

Provide a comprehensive analysis in the following structured format:

╔══════════════════════════════════════════════════════════╗
║     🔍 DATA PROTECTION & PRIVACY RISK ANALYSIS REPORT    ║
╚══════════════════════════════════════════════════════════╝

📌 SITUATION SUMMARY:
[Summarize the situation in 2–3 sentences, identifying the key privacy issue]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

⚠️  RISK LEVEL: [LOW / MEDIUM / HIGH / CRITICAL]
   Justification: [Quantify severity score (n/5) and likelihood score (n/5), rubric available on ALU University Regulations.
   1–2 sentences explaining why this level was assigned based on severity score and likelihood score.]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📋 APPLICABLE LAWS & REGULATIONS:

  From Personal Data Protection Law (UU PDP):
  • [List relevant articles/clauses and what they say]

  From Electronic Information Law (UU ITE & Amendments):
  • [List relevant articles/clauses and what they say]

  From ALU University Regulations:
  • [List relevant sections and what they say]

  (If a source has no relevant clauses, state "No directly applicable clauses found.")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🎯 VIOLATIONS IDENTIFIED:
[List each violation clearly, e.g.:
  1. Unauthorized disclosure of personal data
  2. Lack of consent before data sharing
  3. Breach of university data governance policy]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

⚖️  APPLICABLE SANCTIONS:
[List sanctions from each relevant legal source. Be specific about
 criminal penalties, fines, or institutional disciplinary actions]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

✅ RECOMMENDED ACTIONS:
[Numbered list of concrete steps the university should take immediately
 and long-term. Include both reactive and preventive measures]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📚 DOCUMENT SOURCES USED IN THIS ANALYSIS:
[List all documents that contributed to this analysis, with page references if available]

╚══════════════════════════════════════════════════════════╝

Important instructions:
- Base your analysis ONLY on the context provided above
- If a document provided no relevant content, explicitly state that
- Use formal, professional English throughout
- Be specific when citing articles — include article numbers and clause text
"""

# Convert the prompt string into a LangChain prompt template
prompt_template = ChatPromptTemplate.from_template(PRIVACY_ANALYST_PROMPT)


## Cell 12: Assemble the RAG chain


In [ ]:
# This is the full pipeline:
#   user question
#     → retrieve relevant chunks from all documents
#     → inject context into prompt
#     → send to LLM
#     → parse text output
# ───────────────────────────────────────────────────────────
privacy_agent = (
    {
        # Retrieve context from vector DB using the question
        "context":  lambda x: retrieve_docs(x["question"]),
        # Pass the original question through unchanged
        "question": lambda x: x["question"]
    }
    | prompt_template   # Fill the prompt template with context + question
    | llm               # Send filled prompt to LLaMA 3.3 via Groq API
    | StrOutputParser() # Extract plain text from the LLM response object
)


## Cell 13: Test the agent


In [ ]:
# Run a sample query to verify everything works end-to-end
# ───────────────────────────────────────────────────────────
test_scenario = """
A lecturer sold the phone numbers of students and fellow faculty members
to a third-party marketing company without obtaining consent.
"""

print("🤖 Running privacy analysis agent...\n")
response = privacy_agent.invoke({"question": test_scenario})
print(response)


🤖 Running privacy analysis agent...

╔══════════════════════════════════════════════════════════╗
║     🔍 DATA PROTECTION & PRIVACY RISK ANALYSIS REPORT    ║
╚══════════════════════════════════════════════════════════╝

📌 SITUATION SUMMARY:
A lecturer at the university sold the phone numbers of students and fellow faculty members to a third-party marketing company without obtaining consent, violating the privacy and trust of the individuals involved. This unauthorized disclosure of personal data poses a significant risk to the university's reputation and compliance with data protection regulations. The key privacy issue here is the lack of consent and the misuse of personal data for commercial purposes.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

⚠️  RISK LEVEL: HIGH
   Justification: The severity score is 5/5 due to the sensitive nature of the data (phone numbers) and the potential harm it could cause to the individuals involved, including harassment, identity theft, 

## Cell 14: Streamlit App — ready to deploy


In [ ]:
# Write the Streamlit app code to a file called app.py
streamlit_app_code = '''
# ============================================================
# STREAMLIT WEB APP — Data Privacy Compliance Agent
# Deploy with: streamlit run app.py
# ============================================================

import streamlit as st
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.messages import HumanMessage, AIMessage
import os
import time

# ── Page configuration ──────────────────────────────────────
st.set_page_config(
    page_title="Privacy Compliance Agent",
    page_icon="🔐",
    layout="wide",
    initial_sidebar_state="expanded"
)

# ── Custom CSS for a polished look ──────────────────────────
st.markdown("""
<style>
    /* Main header gradient */
    .main-header {
        background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%);
        padding: 2rem;
        border-radius: 12px;
        text-align: center;
        margin-bottom: 2rem;
        box-shadow: 0 4px 20px rgba(0,0,0,0.3);
    }
    .main-header h1 { color: #e94560; margin: 0; font-size: 2.2rem; }
    .main-header p  { color: #a8b2d8; margin: 0.5rem 0 0 0; font-size: 1rem; }

    /* Risk level badge */
    .risk-critical { background:#ff4444; color:white; padding:6px 16px; border-radius:20px; font-weight:bold; }
    .risk-high     { background:#ff8800; color:white; padding:6px 16px; border-radius:20px; font-weight:bold; }
    .risk-medium   { background:#ffbb00; color:black; padding:6px 16px; border-radius:20px; font-weight:bold; }
    .risk-low      { background:#00bb44; color:white; padding:6px 16px; border-radius:20px; font-weight:bold; }

    /* Result card */
    .result-card {
        background: #0f1923;
        border: 1px solid #1e3a5f;
        border-radius: 10px;
        padding: 1.5rem;
        font-family: monospace;
        white-space: pre-wrap;
        color: #e0e0e0;
        line-height: 1.7;
    }

    /* Sidebar styling */
    .sidebar-section {
        background: #1a1a2e;
        border-radius: 8px;
        padding: 1rem;
        margin-bottom: 1rem;
        border-left: 3px solid #e94560;
    }

    /* Metric boxes */
    .metric-box {
        text-align: center;
        background: #16213e;
        border-radius: 8px;
        padding: 1rem;
        border: 1px solid #0f3460;
    }
    .metric-box .number { font-size: 2rem; font-weight: bold; color: #e94560; }
    .metric-box .label  { font-size: 0.85rem; color: #a8b2d8; }
</style>
""", unsafe_allow_html=True)

# ── Session State Initialization ─────────────────────────────
if "messages" not in st.session_state:
    st.session_state.messages = []
if "last_report" not in st.session_state:
    st.session_state.last_report = None

# ── App header ──────────────────────────────────────────────
st.markdown("""
<div class="main-header">
    <h1>🔐 AI COMPLIANCE AGENT</h1>
    <p>A RAG-based data protection and privacy risk analyst agent for university</p>
    <p>Developed by Yunus P.</p>
</div>
""", unsafe_allow_html=True)

# ── Sidebar: settings and info ──────────────────────────────
with st.sidebar:
    st.markdown("### ⚙️ Configuration")
    api_key = st.text_input("Groq API Key", type="password", placeholder="gsk_...", help="Get a free key at console.groq.com/keys")

    if st.button("🗑️ Clear Chat History"):
        st.session_state.messages = []
        st.rerun()

    st.divider()
    st.markdown("### 📚 Knowledge Base")
    st.markdown("""
    <div class="sidebar-section" style="color: white;">
    This agent analyzes scenarios against:
    <br><br>
    📜 <b>UU PDP</b> — Personal Data Protection Law<br>
    💻 <b>UU ITE</b> — Electronic Information Law<br>
    📝 <b>UU ITE Amendment I</b><br>
    📝 <b>UU ITE Amendment II</b><br>
    🏫 <b>ALU University Regulations</b>
    </div>
    """, unsafe_allow_html=True)

# ── Main layout: two columns ────────────────────────────────
col_left, col_right = st.columns([1, 1.4], gap="large")

with col_left:
    st.markdown("### 📥 Input Scenario")
    examples = {
        "📱 Phone Number Sale": "A lecturer sold student phone numbers to a third-party without consent.",
        "📊 Grade Data Leak": "IT staff accidentally published student grades and ID numbers on the public website.",
        "📷 Unauthorized CCTV": "University installed CCTV in dorm rooms without notifying students."
    }

    for label, text in examples.items():
        if st.button(label, use_container_width=True):
            st.session_state["scenario_input"] = text

    scenario = st.text_area("Describe the privacy scenario:", value=st.session_state.get("scenario_input", ""), height=160, key="scenario_input")
    analyze_btn = st.button("🔍 Analyze Scenario", type="primary", use_container_width=True)

# ── Load models and vector store ────────────────────────────
@st.cache_resource(show_spinner=False)
def load_system(api_key: str):
    os.environ["GROQ_API_KEY"] = api_key
    llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vectorstore = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)
    retriever = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 7})

    def retrieve_docs(question: str) -> str:
        docs = retriever.invoke(question)
        context = ""
        for doc in docs:
            context += f"\\nSource: {doc.metadata.get('source_name')} (Page {doc.metadata.get('page')})\\n{doc.page_content}\\n"
        return context

    # Agent 1: Report Generator
    REPORT_PROMPT = ChatPromptTemplate.from_template("""
    You are an Indonesian University Data Privacy Analyst. Analyze the following:
    CONTEXT: {context}
    QUESTION: {question}
    Provide your analysis in this exact format:

╔════════════════════════════════════════════╗

🔍 DATA PROTECTION & PRIVACY RISK ANALYSIS REPORT

📌 SITUATION SUMMARY:
[Summarize the situation in 2–3 sentences, identifying the key privacy issue]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

⚠️  RISK LEVEL: [LOW / MEDIUM / HIGH / CRITICAL]
   Justification: [Quantify severity score (n/5) and likelihood score (n/5), rubric available on ALU University Regulations.
   1–2 sentences explaining why this level was assigned based on severity score and likelihood score.]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📋 APPLICABLE LAWS & REGULATIONS:
  From Personal Data Protection Law (UU PDP):
  • [List relevant articles/clauses and what they say]
  From Electronic Information Law (UU ITE & Amendments):
  • [List relevant articles/clauses and what they say]
  From ALU University Regulations:
  • [List relevant sections and what they say]

  (If a source has no relevant clauses, state "No directly applicable clauses found.")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🎯 VIOLATIONS IDENTIFIED:
[List each violation clearly, e.g.:
  1. Unauthorized disclosure of personal data
  2. Lack of consent before data sharing
  3. Breach of university data governance policy]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

⚖️  APPLICABLE SANCTIONS:
[List sanctions from each relevant legal source. Be specific about
 criminal penalties, fines, or institutional disciplinary actions]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

✅ RECOMMENDED ACTIONS:
[Numbered list of concrete steps the university should take immediately
 and long-term. Include both reactive and preventive measures]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📚 DOCUMENT SOURCES USED IN THIS ANALYSIS:
[List all documents that contributed to this analysis, with page references if available]

╚════════════════════════════════════════════╝

Base analysis ONLY on provided context. Use formal, professional English.
    """)

    # Agent 2: Chat Assistant
    CHAT_PROMPT = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful Compliance Assistant. Answer questions based ONLY on the provided context: {context}"),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{question}")
    ])

    report_chain = ({"context": lambda x: retrieve_docs(x["question"]), "question": lambda x: x["question"]} | REPORT_PROMPT | llm | StrOutputParser())
    chat_chain = ({"context": lambda x: retrieve_docs(x["question"]), "question": lambda x: x["question"], "chat_history": lambda x: x["chat_history"]} | CHAT_PROMPT | llm | StrOutputParser())

    return report_chain, chat_chain

# ── Tabs for Analysis vs Chat ───────────────────────────────
with col_right:
    tab_report, tab_chat = st.tabs(["📊 Analysis Report", "💬 Discussion Chat"])

    with tab_report:
        if analyze_btn and api_key and scenario:
            with st.spinner("Analyzing..."):
                report_agent, _ = load_system(api_key)
                st.session_state.last_report = report_agent.invoke({"question": scenario})

        if st.session_state.last_report:
            st.markdown(f'<div class="result-card">{st.session_state.last_report}</div>', unsafe_allow_html=True)
        else:
            st.info("Submit a scenario to generate a report.")

    with tab_chat:
        st.caption("Got questions? Ask away! and Let's chat about the details.")
        # Display chat messages
        for message in st.session_state.messages:
            with st.chat_message(message["role"]):
                st.markdown(message["content"])

        if prompt := st.chat_input("Ask me anything about data protection & privacy regulations..."):
            if not api_key:
                st.error("Please enter your API key first.")
            else:
                st.session_state.messages.append({"role": "user", "content": prompt})
                with st.chat_message("user"):
                    st.markdown(prompt)

                with st.chat_message("assistant"):
                    _, chat_agent = load_system(api_key)
                    # Convert history to LangChain format
                    history = []
                    for m in st.session_state.messages[:-1]:
                        if m["role"] == "user": history.append(HumanMessage(content=m["content"]))
                        else: history.append(AIMessage(content=m["content"]))

                    response = chat_agent.invoke({"question": prompt, "chat_history": history})
                    st.markdown(response)
                    st.session_state.messages.append({"role": "assistant", "content": response})

# ── Footer ───────────────────────────────────────────────────
st.divider()
st.markdown('<div style="text-align:center; color:#blue; font-size:0.8rem;">Developed by Yunus P. | Powered by LLaMA 3.3 | For educational and compliance guidance purposes only. Not a substitute for qualified legal counsel</div>', unsafe_allow_html=True)
'''

# Write the app code to disk
with open("app.py", "w") as f:
    f.write(streamlit_app_code)

print("✅ app.py written successfully!")

✅ app.py written successfully!


## Cell 15: Launch Streamlit in Colab via ngrok


In [ ]:
# ngrok creates a public tunnel to your Colab runtime so you
# can open the Streamlit UI in any browser tab.
# Requires a free ngrok account: https://ngrok.com/
# ───────────────────────────────────────────────────────────
import subprocess
import time
from pyngrok import ngrok

# Your ngrok authtoken (get it at: dashboard.ngrok.com/get-started/your-authtoken)
# Add it to Colab secrets as NGROK_TOKEN
NGROK_TOKEN = userdata.get('NGROK_TOKEN')
ngrok.set_auth_token(NGROK_TOKEN)

# Start the Streamlit server in the background (port 8501 is the default)
streamlit_proc = subprocess.Popen(
    ["streamlit", "run", "app.py",
     "--server.port", "8501",
     "--server.headless", "true",   # No browser auto-open in server mode
     "--server.enableCORS", "false" # Required for ngrok to work correctly
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Wait a moment for the server to initialize
print("⏳ Starting Streamlit server...")
time.sleep(4)

# Open a public tunnel on port 8501
public_url = ngrok.connect(8501)
print(f"\n🚀 App is live!")
print(f"🌐 Open in your browser: {public_url}")
print(f"\n   (The link is active as long as this Colab session is running)")

⏳ Starting Streamlit server...

🚀 App is live!
🌐 Open in your browser: NgrokTunnel: "https://shrubbery-habitant-hacksaw.ngrok-free.dev" -> "http://localhost:8501"

   (The link is active as long as this Colab session is running)


In [ ]:
# 1. Disconnect all active tunnels
tunnels = ngrok.get_tunnels()
for tunnel in tunnels:
    print(f"Disconnecting: {tunnel.public_url}")
    ngrok.disconnect(tunnel.public_url)

# 2. Kill the ngrok process entirely
ngrok.kill()

print("✅ Ngrok has been reset. You can now re-run the Launch cell.")

Disconnecting: https://shrubbery-habitant-hacksaw.ngrok-free.dev
✅ Ngrok has been reset. You can now re-run the Launch cell.
